# 後方観測TD(λ)とEligibility Trace

Eligibility Trace は「どの過去状態にどれだけ責任を配るか」を定量化する仕組みです。長期報酬の信用割当をオンラインで扱えるようにするために使います。


## 参考動画
- [対応動画 1](https://www.youtube.com/watch?v=uvxmmIWnJS4)
@[youtube](https://www.youtube.com/watch?v=uvxmmIWnJS4)
- [対応動画 2](https://www.youtube.com/watch?v=E1r0vkv4lkk)
- [対応動画 3](https://www.youtube.com/watch?v=LdL3UrYSDLw)

## 参考リンク
- https://www.youtube.com/watch?v=uvxmmIWnJS4
- https://www.youtube.com/watch?v=E1r0vkv4lkk
- https://www.youtube.com/watch?v=LdL3UrYSDLw


## 背景と目的

同じ TD 誤差でも、直前に頻繁に訪れた状態ほど大きく更新したい場面があります。

Trace の設計（accumulating / replacing）を理解すると、学習安定性をタスクに合わせて調整できます。

2種類の trace を同じ遷移列で比較します。


In [ ]:
gamma = 0.9
lam = 0.8
alpha = 0.1

states = ['s0', 's1', 's0', 's2', 's0']
rewards = [0.1, 0.0, 0.8, -0.2, 0.0]

V_acc = {'s0': 0.0, 's1': 0.0, 's2': 0.0}
V_rep = {'s0': 0.0, 's1': 0.0, 's2': 0.0}
E_acc = {'s0': 0.0, 's1': 0.0, 's2': 0.0}
E_rep = {'s0': 0.0, 's1': 0.0, 's2': 0.0}

for t in range(len(states) - 1):
    s = states[t]
    s_next = states[t + 1]
    r = rewards[t]

    # accumulating trace
    delta_acc = r + gamma * V_acc[s_next] - V_acc[s]
    for k in E_acc:
        E_acc[k] *= gamma * lam
    E_acc[s] += 1.0
    for k in V_acc:
        V_acc[k] += alpha * delta_acc * E_acc[k]

    # replacing trace
    delta_rep = r + gamma * V_rep[s_next] - V_rep[s]
    for k in E_rep:
        E_rep[k] *= gamma * lam
    E_rep[s] = 1.0
    for k in V_rep:
        V_rep[k] += alpha * delta_rep * E_rep[k]

print('Accumulating V =', {k: round(v, 6) for k, v in V_acc.items()})
print('Replacing V    =', {k: round(v, 6) for k, v in V_rep.items()})
print('Accumulating E =', {k: round(v, 6) for k, v in E_acc.items()})
print('Replacing E    =', {k: round(v, 6) for k, v in E_rep.items()})


状態再訪が多い問題では accumulating trace が更新を強めやすく、発散気味な問題では replacing trace が安定しやすい傾向があります。
